# 08b — LangGraph state and human review — Practice

Read the lesson and run the example. Implement each TODO, then run its checks. A NotImplementedError is expected until you write your answer. Try for 20–30 minutes before opening a hint; open the solution only after recording an attempt. Copy this notebook into work/ to keep your own version.

**Prerequisites:** earlier lessons in the course order. **Environment:** Install requirements/agents.txt. No model or API key is needed.

[Stage guide](../stages/08.md) · [Course map](../PLAN.md) · [Project](../../projects/stage08/README.md)

**Notebook time:** 2–4 hours for an initial pass; project, reading, retrieval practice, and independent transfer use the rest of the stage budget.

Learning objectives:

- Translate a workflow into nodes and edges
- Use explicit typed state
- Compare framework and plain-Python behavior


In [ ]:
from pathlib import Path
import sys

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datasets" / "financials.csv").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from inside the learning repository.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

def expect_error(error_type, operation):
    try:
        operation()
    except error_type:
        return
    raise AssertionError(f"Expected {error_type.__name__}")

print("Course root:", ROOT)


## Lesson

LangGraph represents orchestration as a graph of state transitions. Nodes read state and return updates; edges define the next step. Conditional edges route using explicit outcomes. A reducer defines how parallel updates combine. LangChain’s tool and message abstractions can simplify interfaces, but domain validation must still live in application code.

Checkpointing stores progress so a run can resume. Persistence requires an appropriately durable backend and a stable run/thread identity. Human-in-the-loop interrupts suspend before a sensitive action and resume with a decision. A saved approval must identify the exact proposed action; a generic approved boolean can be reused incorrectly.

Start with one deterministic graph and compare it against a plain Python workflow using the same fixtures. Only add model routing after you can test each transition. A graph reaching END proves termination, not that the final answer is correct. Evaluate tool selection, arguments, permissions, costs, and answer support separately.


## Worked example

Predict the output before running. Change one input and explain the result.


In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
class State(TypedDict, total=False):
    question: str
    route: str
    answer: str


## Exercise 1: Route state

Return route=calculator when question contains the word ratio (case-insensitive), else search. Return only the update.

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def route_node(state):
    raise NotImplementedError


In [ ]:
assert route_node({"question": "calculate ratio"}) == {"route": "calculator"}
print("Exercise 1: checks passed")


<details><summary>Hint — open after trying</summary>

A node can return a partial state update.

</details>


## Exercise 2: Compile graph

Build START→route→conditional calculator/search→END. Calculator returns answer="calculator selected" and search returns "search selected".

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def build_graph():
    raise NotImplementedError


In [ ]:
graph = build_graph()
assert graph.invoke({"question": "revenue ratio"})["answer"] == "calculator selected"
assert graph.invoke({"question": "find policy"})["answer"] == "search selected"
print("Exercise 2: checks passed")


<details><summary>Hint — open after trying</summary>

Keep node names and state keys distinct.

</details>


## Exercise 3: Approval binding

Return SHA-256 of canonical JSON containing actor, action, arguments. Same content in different key order must match.

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def approval_fingerprint(actor, action, arguments):
    raise NotImplementedError


In [ ]:
assert approval_fingerprint("a", "write", {"x": 1, "y": 2}) == approval_fingerprint("a", "write", {"y": 2, "x": 1})
assert approval_fingerprint("a", "write", {"x": 1}) != approval_fingerprint("b", "write", {"x": 1})
print("Exercise 3: checks passed")


<details><summary>Hint — open after trying</summary>

Hash a canonical representation.

</details>


## Independent transfer

Run Stage 08’s checkpoint/interrupt example. Resume one approved action, reject a changed action, and verify two thread IDs cannot exchange state. Compare one graph versus a two-worker design on measured cost and correctness.

Record your implementation, errors, measurements, and explanation in work/. See the project’s ANSWERS.md after attempting the brief.


## Explain without code

1. What must persistence isolate?

2. Does a framework remove authorization work?


## Reading and review

- [Primary reference 1](https://docs.langchain.com/oss/python/langgraph/interrupts)

After 2, 7, and 30 days: explain the concept from memory, solve a changed-input version, and log any gaps in progress.md.
